# Structured Output with Strands Agents

## Overview

When you call a Strands agent, it returns free-form text by default. That's fine for chatbots, but when the agent's output needs to feed into downstream code — an API response, a database write, a UI component — you need structured, typed, validated data.

In this tutorial, we'll walk you through how to use structured output to get back validated Python objects instead of strings.

<div style="text-align:left">
    <img src="images/architecture.png" width="85%" />
</div>

| Feature | Description |
|---------|-------------|
| **Flat Pydantic Models** | Define a model, pass it to the agent, and access typed results |
| **Complex Schemas** | Nested models, lists, optional fields, validators, and enums |
| **Validation & Self-Correction** | Automatic retry when validation fails, exception handling, custom forcing prompt |
| **Tools + Structured Output** | Combine tool use with structured results |
| **Streaming** | Understand where structured output fits in a streaming workflow |

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock, [guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access-modify.html)
* Familiarity with Strands Agents basics [(see Tutorial 01)](../01-first-agent/)

Let's now install the required packages

In [ ]:
!pip install -r requirements.txt -q

### Importing dependency packages

Now let's import the dependency packages

In [ ]:
from enum import Enum
from typing import List, Literal, Optional

from pydantic import BaseModel, Field, field_validator

from strands import Agent
from strands.types.exceptions import StructuredOutputException

In [ ]:
# Model used throughout this tutorial
MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

## 1. Your First Structured Output

Let's start with the simplest use case: you want the agent to return data in a specific shape instead of free-form text.

**The pattern:**
1. Define a Pydantic `BaseModel` with the fields you need
2. Pass it as `structured_output_model` when calling the agent
3. Access the typed result via `result.structured_output`

### Defining a Pydantic Model

This is a plain Pydantic model — the same kind you'd use for an API schema or database record. The `Field(description=...)` helps the LLM understand what each field expects.

In [ ]:
class MovieReview(BaseModel):
    """A structured movie review."""

    title: str = Field(description="The movie title")
    rating: int = Field(description="Rating from 1 to 10")
    summary: str = Field(description="A brief one-sentence summary of the review")
    recommend: bool = Field(description="Whether you would recommend this movie")

### Synchronous call

Let's pass `structured_output_model` when calling the agent. The result is a validated `MovieReview` instance — not a string, not a dict.

> **Note:** The exact field values in the output will vary between runs since the LLM generates content dynamically. The structure and types will always match your Pydantic model.

In [ ]:
agent = Agent(model=MODEL_ID)

result = agent("Review the movie Inception by Christopher Nolan", structured_output_model=MovieReview)

# Access the typed result
review = result.structured_output
print(f"Type: {type(review).__name__}")
print(f"Title: {review.title}")
print(f"Rating: {review.rating}/10")
print(f"Summary: {review.summary}")
print(f"Recommend: {review.recommend}")

### Asynchronous call

The same pattern works with `invoke_async()` for async workflows.

In [ ]:
async def get_review_async():
    agent = Agent(model=MODEL_ID)
    result = await agent.invoke_async(
        "Review the movie The Matrix",
        structured_output_model=MovieReview,
    )
    review = result.structured_output
    print(f"[Async] {review.title}: {review.rating}/10 — {review.summary}")


await get_review_async()

## 2. Complex Schemas

Real-world data isn't flat. Let's see how structured output handles the same Pydantic features you'd use in production: nested models, lists, optional fields, constrained values, and enums.

### Nested models and lists

Let's define sub-models and compose them. The SDK converts the full schema — including nested `$ref` types — into a tool specification the LLM can understand.

In [ ]:
class Address(BaseModel):
    """A physical address."""

    street: str = Field(description="Street address")
    city: str = Field(description="City name")
    state: str = Field(description="State or province")
    country: str = Field(description="Country")


class Skill(BaseModel):
    """A professional skill with proficiency level."""

    name: str = Field(description="Skill name")
    years_experience: int = Field(description="Years of experience", ge=0)


class Candidate(BaseModel):
    """A job candidate profile."""

    name: str = Field(description="Full name")
    address: Address = Field(description="Home address")
    skills: List[Skill] = Field(description="List of professional skills")
    bio: Optional[str] = Field(default=None, description="Short bio, if available")

In [ ]:
agent = Agent(model=MODEL_ID)

result = agent(
    "Create a profile for a fictional senior software engineer based in Seattle with 3 skills",
    structured_output_model=Candidate,
)

candidate = result.structured_output
print(f"Name: {candidate.name}")
print(f"Location: {candidate.address.city}, {candidate.address.state}")
print(f"Bio: {candidate.bio}")
print("Skills:")
for skill in candidate.skills:
    print(f"  - {skill.name}: {skill.years_experience} years")

### Field constraints and enums

We can use Pydantic's `Field` constraints (`ge`, `le`, `min_length`, etc.) and `Literal`/`Enum` types to restrict valid values. The LLM sees these constraints in the tool schema and respects them.

In [ ]:
class Priority(str, Enum):
    """Task priority levels."""

    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"


class TicketAnalysis(BaseModel):
    """Analysis of a customer support ticket."""

    category: str = Field(description="Issue category", min_length=1)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="Customer sentiment")
    priority: Priority = Field(description="Ticket priority")
    confidence: float = Field(description="Confidence score", ge=0.0, le=1.0)
    summary: str = Field(description="Brief summary of the issue")

In [ ]:
agent = Agent(model=MODEL_ID)

ticket_text = """
I've been trying to reset my password for 3 days now and the reset email never arrives.
I've checked spam. This is blocking my entire team from accessing the dashboard.
We're paying for the enterprise plan and this level of service is unacceptable.
"""

result = agent(
    f"Analyze this support ticket:\n{ticket_text}",
    structured_output_model=TicketAnalysis,
)

analysis = result.structured_output
print(f"Category: {analysis.category}")
print(f"Sentiment: {analysis.sentiment}")
print(f"Priority: {analysis.priority.value}")
print(f"Confidence: {analysis.confidence:.0%}")
print(f"Summary: {analysis.summary}")

## 3. Validation & Self-Correction

What happens when the LLM returns data that doesn't pass Pydantic validation?

The SDK handles this automatically:
1. The LLM calls the structured output tool with its data
2. Pydantic validates the data — if it fails, the SDK formats a per-field error message
3. The error is sent back to the LLM as a tool result
4. The LLM reads the error and self-corrects on the next attempt

This happens within the normal agent loop — no extra code needed.

### Triggering the retry loop

Let's define a model with a strict `@field_validator` that the LLM is likely to violate on its first attempt. This lets us observe the self-correction cycle.

In [ ]:
class StrictCode(BaseModel):
    """A product code that must follow a strict format."""

    product_name: str = Field(description="Name of the product")
    product_code: str = Field(description="Product code in format: 3 uppercase letters, dash, 4 digits (e.g., ABC-1234)")

    @field_validator("product_code")
    @classmethod
    def validate_code_format(cls, v: str) -> str:
        """Enforce the exact format: AAA-0000."""
        import re

        if not re.match(r"^[A-Z]{3}-\d{4}$", v):
            raise ValueError(f"Product code '{v}' must match format AAA-0000 (3 uppercase letters, dash, 4 digits)")
        return v

In [ ]:
agent = Agent(model=MODEL_ID)

# The LLM may initially produce a code like "laptop-001" — the validator will reject it,
# the error goes back to the LLM, and it self-corrects to something like "LAP-0001".
# You can enable debug logging (logging.getLogger("strands").setLevel(logging.DEBUG))
# to see the validation error and retry in the agent's logs.
result = agent(
    "Generate a product entry for a wireless keyboard",
    structured_output_model=StrictCode,
)

product = result.structured_output
print(f"Product: {product.product_name}")
print(f"Code: {product.product_code}")

### Handling StructuredOutputException

If the LLM fails to produce valid output even after the SDK forces it, a `StructuredOutputException` is raised. Always wrap structured output calls in a try/except for production code.

In [ ]:
agent = Agent(model=MODEL_ID)

try:
    result = agent(
        "Generate a product entry for a USB hub",
        structured_output_model=StrictCode,
    )
    product = result.structured_output
    print(f"Success: {product.product_name} — {product.product_code}")
except StructuredOutputException as e:
    print(f"Structured output failed: {e}")
    print("Fallback: use the raw text response or retry with a different prompt")

### Custom forcing prompt

When the LLM doesn't call the structured output tool on its own, the SDK sends a forcing prompt to nudge it. The default is:

> *"You must format the previous response as structured output."*

You can customize this with `structured_output_prompt` to give the LLM more specific guidance for your schema.

In [ ]:
agent = Agent(model=MODEL_ID)

result = agent(
    "Generate a product entry for a mechanical keyboard",
    structured_output_model=StrictCode,
    structured_output_prompt=(
        "Format your response using the structured output tool. "
        "The product_code MUST be exactly 3 uppercase letters, a dash, then 4 digits (e.g., MKB-0001)."
    ),
)

product = result.structured_output
print(f"Product: {product.product_name}")
print(f"Code: {product.product_code}")

## 4. Combining Tools with Structured Output

In practice, agents don't just format text — they use tools to gather information first, then structure the result. The structured output tool coexists with regular tools. The LLM uses regular tools to do work, then calls the structured output tool last to return the final answer.

Let's use the `calculator` tool from `strands-agents-tools` so the agent can perform calculations before returning a structured result.

In [ ]:
from strands_tools import calculator


class InvestmentAnalysis(BaseModel):
    """Analysis of an investment calculation."""

    initial_amount: float = Field(description="Initial investment amount in dollars")
    annual_rate: float = Field(description="Annual interest rate as a percentage")
    years: int = Field(description="Investment period in years")
    final_amount: float = Field(description="Calculated final amount after compound interest")
    total_return: float = Field(description="Total return as a percentage")
    recommendation: str = Field(description="Brief investment recommendation")

In [ ]:
agent = Agent(
    model=MODEL_ID,
    tools=[calculator],
    system_prompt="You are a financial analyst. Use the calculator tool for all math operations.",
)

result = agent(
    "If I invest $10,000 at 7% annual compound interest for 15 years, what will I have? Analyze this investment.",
    structured_output_model=InvestmentAnalysis,
)

analysis = result.structured_output
print(f"Initial: ${analysis.initial_amount:,.2f}")
print(f"Rate: {analysis.annual_rate}%")
print(f"Period: {analysis.years} years")
print(f"Final: ${analysis.final_amount:,.2f}")
print(f"Return: {analysis.total_return:.1f}%")
print(f"Recommendation: {analysis.recommendation}")

The agent used the `calculator` tool to compute compound interest, then called the structured output tool to return the result as a validated `InvestmentAnalysis` object. The tools and structured output work together seamlessly.

## 5. Streaming with Structured Output

When building streaming UIs, you want to show progress while the agent works. With structured output:

* **Text chunks may stream** as the agent reasons and uses tools
* **Structured output is NOT available incrementally** — it appears only in the **final event**

This is because the structured output must be fully validated before it's returned.

> **Note:** When structured output is active, the LLM primarily interacts via tool calls (the structured output tool). You may see minimal or no streaming text before the final structured result — this is expected behavior.

In [ ]:
class BookSummary(BaseModel):
    """A structured book summary."""

    title: str = Field(description="Book title")
    author: str = Field(description="Author name")
    genre: str = Field(description="Primary genre")
    themes: List[str] = Field(description="Key themes")
    one_liner: str = Field(description="One-sentence summary")

In [ ]:
async def stream_structured_output():
    agent = Agent(model=MODEL_ID)

    print("Streaming events:")
    print("-" * 40)

    async for event in agent.stream_async(
        "Summarize the book 'The Hitchhiker's Guide to the Galaxy' by Douglas Adams",
        structured_output_model=BookSummary,
    ):
        # Text chunks arrive during processing
        if "data" in event:
            print(f"[stream] {event['data']}", end="", flush=True)

        # Structured output arrives only in the final result
        if "result" in event:
            print("\n" + "-" * 40)
            print("Final structured output:")
            summary = event["result"].structured_output
            print(f"  Title: {summary.title}")
            print(f"  Author: {summary.author}")
            print(f"  Genre: {summary.genre}")
            print(f"  Themes: {', '.join(summary.themes)}")
            print(f"  Summary: {summary.one_liner}")


await stream_structured_output()

The validated `BookSummary` object only appears in the final `result` event. Keep this in mind when designing your UI — you can show a loading indicator for the structured data while streaming text progress.

## Key Takeaways

You've learned how to get reliable, typed data from Strands agents:

1. **Basic structured output** — Define a Pydantic model, pass it to the agent, get a validated object back
2. **Complex schemas** — Nested models, lists, optional fields, field constraints, and enums all work
3. **Validation & self-correction** — The SDK automatically retries when validation fails, and you can catch `StructuredOutputException` as a fallback
4. **Tools + structured output** — Agents use tools to gather data, then return structured results
5. **Streaming** — Structured output appears in the final event, not incrementally

### Tips for production use

* Use `structured_output_model=` (not the deprecated `agent.structured_output()` method)
* Add `Field(description=...)` to help the LLM understand your schema
* Always wrap production calls in `try/except StructuredOutputException`
* Use `structured_output_prompt` to improve success rates for strict schemas
* Structured output works with both sync (`agent()`) and async (`invoke_async()`, `stream_async()`)